In [26]:
!pip install playwright

In [27]:
!playwright install

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Missing libraries:                                   ║
║     libwoff2dec.so.1.0.2                             ║
║     libgstgl-1.0.so.0                                ║
║     libgstcodecparsers-1.0.so.0                      ║
║     libavif.so.13                                    ║
║     libharfbuzz-icu.so.0                             ║
║     libenchant-2.so.2                                ║
║     libsecret-1.so.0                                 ║
║     libhyphen.so.0                                   ║
║     libmanette-0.2.so.0                              ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.11/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:216:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:105

In [31]:
from playwright.async_api import async_playwright
import asyncio
import pandas as pd

URL = "https://foodsuppliers.ru/companies/kofe-moskva"

async def get_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URL)
        await page.wait_for_selector("div.content-list-item__txt")
        company_blocks = await page.query_selector_all("div.content-list-item__txt")
        links = await page.query_selector_all("a.title-site--h3")
        hrefs = [await link.get_attribute("href") for link in links]
        titles = [await link.inner_text() for link in links]
        companies_data = []
        for i, block in enumerate(company_blocks):
            name = titles[i]
            description = await block.query_selector("div.field-item.even p")
            description = await description.inner_text()
            address = await block.query_selector("p.text--col1")
            address = await address.inner_text() if address else "N/A"
            link = f'https://foodsuppliers.ru{hrefs[i]}'
            companies_data.append({
                "Название": name,
                "Описание": description,
                "Адрес": address,
                "Ссылка": link
            })
        df = pd.DataFrame(companies_data)
        df.to_csv("companies_data.csv", index=False, encoding="utf-8")
        await browser.close()

await get_data()


In [32]:
database = pd.read_csv("companies_data.csv")
display(database)

,Название,Описание,Адрес,Ссылка
0,Solo Coffee,Производитель свежеобжаренного кофе.,"Москва, Каширское шоссе, д.49, стр.22",https://foodsuppliers.ru/company/solo-coffee
1,Эллада,Поставщики продуктов из Греции с 1997 года.,"115280, г. Москва, ул. Автозаводская, д. 23А, ...",https://foodsuppliers.ru/company/ellada-0
2,Весом.ру,Нашей специализацией на сегодня являются бакал...,"Россия, Москва, поселение Сосенское, Калужское...",https://foodsuppliers.ru/company/vesomru
3,Купинаразвес ру,В интернет-магазина KUPINARAZVES.RU вы всегда ...,"Россия, Москва, проспект 60-летия Октября, 18к2",https://foodsuppliers.ru/company/kupinarazves-ru
4,Орешкин Дом,Орехи и сухофрукты – это незаменимые продукты ...,"Россия, Москва, Варшавское шоссе, 26с6",https://foodsuppliers.ru/company/oreshkin-dom
5,Царь Миндаль,"Царь Миндаль это не просто магазин, где можно ...","Россия, Москва, улица Искры, 31к1",https://foodsuppliers.ru/company/car-mindal
6,BonSapore,Компания BonSapore является официальным дистри...,"Россия, Москва, Кольская улица, 2к4",https://foodsuppliers.ru/company/bonsapore
7,Tam-Tam,Компания «Там-Там» была образована в 1996 году...,проспект Мира д 102 стр 1 оф 204,https://foodsuppliers.ru/company/tam-tam
8,Шишкин Лес Торг,Компания «Шишкин Лес» основана в 1998 году. За...,"108833 Московская область, Москва, МОСКВА ГОРО...",https://foodsuppliers.ru/company/shishkin-les-...
9,Эвотор,"""На-полке"" - это онлайн-платформа, помогающая ...","115280, Москва, ул. Ленинская Слобода д. 26, с5",https://foodsuppliers.ru/company/evotor


In [50]:

async def get_info(page, URL):
    await page.goto(URL)

    await page.wait_for_selector("div.about-project")
    about_project_block = await page.query_selector("div.about-project")
    paragraphs = await about_project_block.query_selector_all("p")
    all_paragraphs_text = []
    for p in paragraphs:
        text = await p.inner_text()
        all_paragraphs_text.append(text)
    about_project_info = "\n".join(all_paragraphs_text)

    goods_and_services_block = await page.query_selector("div.about-project")
    goods_and_services = []
    if goods_and_services_block:
        await page.wait_for_selector("ul li span.lineage-item", timeout=5000)
        items = await page.query_selector_all("ul li span.lineage-item")
        for item in items:
            text = await item.inner_text()
            goods_and_services.append(text)
    goods_and_services_info = ", ".join(goods_and_services)

    await page.wait_for_selector("div.section-st-block")
    contacts_data = {}
    contacts_block = await page.query_selector("div.section-st-block#section-st-block5")
    contact_items = await contacts_block.query_selector_all("div.content-contact-item")
    for item in contact_items:
        title_element = await item.query_selector("div.content-contact-item__tt")
        title = await title_element.inner_text()
        block_element = await item.query_selector("div.content-contact-item__block")
        block_text = await block_element.inner_text()
        contacts_data[title] = block_text.strip()

    return {
        "Ссылка": URL,
        "Информация о компании": about_project_info,
        "Товары и услуги": goods_and_services_info,
        **contacts_data
        }


async def get_data():
  all_data = []
  async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()
    links = database["Ссылка"].tolist()
    for url in links:
      data = await get_info(page, url)
      all_data.append(data)
    df = pd.DataFrame(all_data)
    df.to_csv("all_companies_data.csv", index=False, encoding="utf-8")
    await browser.close()

await get_data()

  0%|          | 0/15 [46:53<?, ?it/s]


In [51]:
database_full = pd.read_csv("all_companies_data.csv")
database_full["Информация о компании"] = database_full["Информация о компании"].str.replace("\n", "")
database_full["Телефон"] = database_full["Телефон"].str.replace("\n", "<br>", regex=False)
display(database_full)


,Ссылка,Информация о компании,Товары и услуги,Населенный пункт,Тип компании,Адрес,Полное название,Режим работы,ИНН,Телефон,Электронная почта,Веб-сайт
0,https://foodsuppliers.ru/company/solo-coffee,Мы компания Solo Coffee — занимаемся обжаркой ...,Кофе,Москва (Города федерального значения),Производитель,"Москва, Каширское шоссе, д.49, стр.22","ООО ""СОЛО КОФЕ""",9-18,5.003141e+09,+79015859277,info@solocoffee.su,solocoffee.su
1,https://foodsuppliers.ru/company/ellada-0,Поставщики продуктов из Греции с 1997 года. Го...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Поставщик,"115280, г. Москва, ул. Автозаводская, д. 23А, ...",ООО «ЭЛЛАДА»,"пн-пт , с 9:00 до 18:00",7.714920e+09,89151485822,a.borisikhin@delphi-food.ru,b2btrade.ru
2,https://foodsuppliers.ru/company/vesomru,Мы работаем с 2011 года. Для нас очень важно б...,"Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Россия, Москва, поселение Сосенское, Калужское...",NaN,NaN,NaN,8 (499) 500-14-34,zakaz@vesom.ru\nzakupka@vesom.ru\nad@vesom.ru,https://vesom.ru/
3,https://foodsuppliers.ru/company/kupinarazves-ru,В интернет-магазина KUPINARAZVES.RU вы всегда ...,"Кофе, Чай, Бобовые, Киноа, Нут, Рис, Арахис, Б...",Москва (Города федерального значения),Поставщик,"Россия, Москва, проспект 60-летия Октября, 18к2",NaN,NaN,NaN,8-985-099-20-30<br>8 (495) 120-90-99,info@kupinarazves.ru,https://kupinarazves.ru/
4,https://foodsuppliers.ru/company/oreshkin-dom,Орехи и сухофрукты – это незаменимые продукты ...,"Мед, Кофе, Чай, Бобовые, Арахис, Бразильский о...",Москва (Города федерального значения),Поставщик,"Россия, Москва, Варшавское шоссе, 26с6",NaN,NaN,NaN,+7 (495) 203-00-09<br>+7 (926) 004-03-22,info@oreh-dom.ru,https://oreh-dom.ru/
5,https://foodsuppliers.ru/company/car-mindal,"Царь Миндаль это не просто магазин, где можно ...","Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Россия, Москва, улица Искры, 31к1",NaN,NaN,NaN,+7 (812) 565-72-74,sale@tsarmindal.ru,https://msk.tsarmindal.ru/
6,https://foodsuppliers.ru/company/bonsapore,Компания BonSapore является официальным дистри...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Дистрибьютор,"Россия, Москва, Кольская улица, 2к4",NaN,NaN,NaN,+7 (495) 295-05-14,info@bonsapore.ru,https://bonsapore.ru/
7,https://foodsuppliers.ru/company/tam-tam,Компания «Там-Там» была образована в 1996 году...,"Кофе, Фруктовые консервы, Арахис, Кешью, Вялен...",Москва (Города федерального значения),Дистрибьютор,проспект Мира д 102 стр 1 оф 204,NaN,NaN,NaN,8 (495) 742-02-80<br>8 (495) 742-10-32,son0104@mail.ru\nkoliatamtam@mail.ru,http://www.tamtamfoods.ru/
8,https://foodsuppliers.ru/company/shishkin-les-...,Компания «Шишкин Лес» основана в 1998 году. За...,"Сахар, Кофе, Чай, Вода, Питьевая вода",Москва (Города федерального значения),Производитель,"108833 Московская область, Москва, МОСКВА ГОРО...",NaN,NaN,NaN,+7(495) 258-25-78,info@cone-forest.ru,https://cone-forest.ru/
9,https://foodsuppliers.ru/company/evotor,"""На-полке"" - это онлайн-платформа, помогающая ...","Дрожжи, Макароны, Мука, Приправы, Сахар, Соль,...",Москва (Города федерального значения),Поставщик,"115280, Москва, ул. Ленинская Слобода д. 26, с5",NaN,NaN,NaN,8 800 222-04-86,zakaz@napolke.ru,https://napolke.ru/


In [52]:
merged_df = pd.merge(database, database_full, left_on=database.columns[-1], right_on=database_full.columns[0])
merged_df = merged_df.drop(columns = ["Адрес_x"]) #убираем дублирующиеся колонки
merged_df = merged_df.rename(columns={'Адрес_y': 'Адрес'})
merged_df['Адрес'] = merged_df['Адрес'].apply(lambda address: 'Москва' + address.lower().split('москва')[1] if 'москва' in address.lower() else address)
display(merged_df)

,Название,Описание,Ссылка,Информация о компании,Товары и услуги,Населенный пункт,Тип компании,Адрес,Полное название,Режим работы,ИНН,Телефон,Электронная почта,Веб-сайт
0,Solo Coffee,Производитель свежеобжаренного кофе.,https://foodsuppliers.ru/company/solo-coffee,Мы компания Solo Coffee — занимаемся обжаркой ...,Кофе,Москва (Города федерального значения),Производитель,"Москва, каширское шоссе, д.49, стр.22","ООО ""СОЛО КОФЕ""",9-18,5.003141e+09,+79015859277,info@solocoffee.su,solocoffee.su
1,Эллада,Поставщики продуктов из Греции с 1997 года.,https://foodsuppliers.ru/company/ellada-0,Поставщики продуктов из Греции с 1997 года. Го...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Поставщик,"Москва, ул. автозаводская, д. 23а, к. 2, этаж ...",ООО «ЭЛЛАДА»,"пн-пт , с 9:00 до 18:00",7.714920e+09,89151485822,a.borisikhin@delphi-food.ru,b2btrade.ru
2,Весом.ру,Нашей специализацией на сегодня являются бакал...,https://foodsuppliers.ru/company/vesomru,Мы работаем с 2011 года. Для нас очень важно б...,"Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Москва, поселение сосенское, калужское шоссе, ...",NaN,NaN,NaN,8 (499) 500-14-34,zakaz@vesom.ru\nzakupka@vesom.ru\nad@vesom.ru,https://vesom.ru/
3,Купинаразвес ру,В интернет-магазина KUPINARAZVES.RU вы всегда ...,https://foodsuppliers.ru/company/kupinarazves-ru,В интернет-магазина KUPINARAZVES.RU вы всегда ...,"Кофе, Чай, Бобовые, Киноа, Нут, Рис, Арахис, Б...",Москва (Города федерального значения),Поставщик,"Москва, проспект 60-летия октября, 18к2",NaN,NaN,NaN,8-985-099-20-30<br>8 (495) 120-90-99,info@kupinarazves.ru,https://kupinarazves.ru/
4,Орешкин Дом,Орехи и сухофрукты – это незаменимые продукты ...,https://foodsuppliers.ru/company/oreshkin-dom,Орехи и сухофрукты – это незаменимые продукты ...,"Мед, Кофе, Чай, Бобовые, Арахис, Бразильский о...",Москва (Города федерального значения),Поставщик,"Москва, варшавское шоссе, 26с6",NaN,NaN,NaN,+7 (495) 203-00-09<br>+7 (926) 004-03-22,info@oreh-dom.ru,https://oreh-dom.ru/
5,Царь Миндаль,"Царь Миндаль это не просто магазин, где можно ...",https://foodsuppliers.ru/company/car-mindal,"Царь Миндаль это не просто магазин, где можно ...","Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Москва, улица искры, 31к1",NaN,NaN,NaN,+7 (812) 565-72-74,sale@tsarmindal.ru,https://msk.tsarmindal.ru/
6,BonSapore,Компания BonSapore является официальным дистри...,https://foodsuppliers.ru/company/bonsapore,Компания BonSapore является официальным дистри...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Дистрибьютор,"Москва, кольская улица, 2к4",NaN,NaN,NaN,+7 (495) 295-05-14,info@bonsapore.ru,https://bonsapore.ru/
7,Tam-Tam,Компания «Там-Там» была образована в 1996 году...,https://foodsuppliers.ru/company/tam-tam,Компания «Там-Там» была образована в 1996 году...,"Кофе, Фруктовые консервы, Арахис, Кешью, Вялен...",Москва (Города федерального значения),Дистрибьютор,проспект Мира д 102 стр 1 оф 204,NaN,NaN,NaN,8 (495) 742-02-80<br>8 (495) 742-10-32,son0104@mail.ru\nkoliatamtam@mail.ru,http://www.tamtamfoods.ru/
8,Шишкин Лес Торг,Компания «Шишкин Лес» основана в 1998 году. За...,https://foodsuppliers.ru/company/shishkin-les-...,Компания «Шишкин Лес» основана в 1998 году. За...,"Сахар, Кофе, Чай, Вода, Питьевая вода",Москва (Города федерального значения),Производитель,"Москва,",NaN,NaN,NaN,+7(495) 258-25-78,info@cone-forest.ru,https://cone-forest.ru/
9,Эвотор,"""На-полке"" - это онлайн-платформа, помогающая ...",https://foodsuppliers.ru/company/evotor,"""На-полке"" - это онлайн-платформа, помогающая ...","Дрожжи, Макароны, Мука, Приправы, Сахар, Соль,...",Москва (Города федерального значения),Поставщик,"Москва, ул. ленинская слобода д. 26, с5",NaN,NaN,NaN,8 800 222-04-86,zakaz@napolke.ru,https://napolke.ru/


In [54]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from tqdm import tqdm
import time

geolocator = Nominatim(user_agent="providers_map")
merged_df[["Latitude", "Longitude"]] = None, None
for i, row in merged_df.iterrows():
    location = geolocator.geocode(row["Адрес"])
    if location:
        merged_df.at[i, "Latitude"] = location.latitude
        merged_df.at[i, "Longitude"] = location.longitude
    time.sleep(1)
display(merged_df)

,Название,Описание,Ссылка,Информация о компании,Товары и услуги,Населенный пункт,Тип компании,Адрес,Полное название,Режим работы,ИНН,Телефон,Электронная почта,Веб-сайт,Latitude,Longitude
0,Solo Coffee,Производитель свежеобжаренного кофе.,https://foodsuppliers.ru/company/solo-coffee,Мы компания Solo Coffee — занимаемся обжаркой ...,Кофе,Москва (Города федерального значения),Производитель,"Москва, каширское шоссе, д.49, стр.22","ООО ""СОЛО КОФЕ""",9-18,5.003141e+09,+79015859277,info@solocoffee.su,solocoffee.su,None,None
1,Эллада,Поставщики продуктов из Греции с 1997 года.,https://foodsuppliers.ru/company/ellada-0,Поставщики продуктов из Греции с 1997 года. Го...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Поставщик,"Москва, ул. автозаводская, д. 23а, к. 2, этаж ...",ООО «ЭЛЛАДА»,"пн-пт , с 9:00 до 18:00",7.714920e+09,89151485822,a.borisikhin@delphi-food.ru,b2btrade.ru,None,None
2,Весом.ру,Нашей специализацией на сегодня являются бакал...,https://foodsuppliers.ru/company/vesomru,Мы работаем с 2011 года. Для нас очень важно б...,"Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Москва, поселение сосенское, калужское шоссе, ...",NaN,NaN,NaN,8 (499) 500-14-34,zakaz@vesom.ru\nzakupka@vesom.ru\nad@vesom.ru,https://vesom.ru/,None,None
3,Купинаразвес ру,В интернет-магазина KUPINARAZVES.RU вы всегда ...,https://foodsuppliers.ru/company/kupinarazves-ru,В интернет-магазина KUPINARAZVES.RU вы всегда ...,"Кофе, Чай, Бобовые, Киноа, Нут, Рис, Арахис, Б...",Москва (Города федерального значения),Поставщик,"Москва, проспект 60-летия октября, 18к2",NaN,NaN,NaN,8-985-099-20-30<br>8 (495) 120-90-99,info@kupinarazves.ru,https://kupinarazves.ru/,55.689434,37.573364
4,Орешкин Дом,Орехи и сухофрукты – это незаменимые продукты ...,https://foodsuppliers.ru/company/oreshkin-dom,Орехи и сухофрукты – это незаменимые продукты ...,"Мед, Кофе, Чай, Бобовые, Арахис, Бразильский о...",Москва (Города федерального значения),Поставщик,"Москва, варшавское шоссе, 26с6",NaN,NaN,NaN,+7 (495) 203-00-09<br>+7 (926) 004-03-22,info@oreh-dom.ru,https://oreh-dom.ru/,55.683921,37.621498
5,Царь Миндаль,"Царь Миндаль это не просто магазин, где можно ...",https://foodsuppliers.ru/company/car-mindal,"Царь Миндаль это не просто магазин, где можно ...","Кофе, Чай, Арахис, Бразильский орех, Грецкие о...",Москва (Города федерального значения),Поставщик,"Москва, улица искры, 31к1",NaN,NaN,NaN,+7 (812) 565-72-74,sale@tsarmindal.ru,https://msk.tsarmindal.ru/,55.864258,37.651167
6,BonSapore,Компания BonSapore является официальным дистри...,https://foodsuppliers.ru/company/bonsapore,Компания BonSapore является официальным дистри...,"Макароны, Приправы, Растительное масло, Уксус,...",Москва (Города федерального значения),Дистрибьютор,"Москва, кольская улица, 2к4",NaN,NaN,NaN,+7 (495) 295-05-14,info@bonsapore.ru,https://bonsapore.ru/,55.858304,37.655067
7,Tam-Tam,Компания «Там-Там» была образована в 1996 году...,https://foodsuppliers.ru/company/tam-tam,Компания «Там-Там» была образована в 1996 году...,"Кофе, Фруктовые консервы, Арахис, Кешью, Вялен...",Москва (Города федерального значения),Дистрибьютор,проспект Мира д 102 стр 1 оф 204,NaN,NaN,NaN,8 (495) 742-02-80<br>8 (495) 742-10-32,son0104@mail.ru\nkoliatamtam@mail.ru,http://www.tamtamfoods.ru/,None,None
8,Шишкин Лес Торг,Компания «Шишкин Лес» основана в 1998 году. За...,https://foodsuppliers.ru/company/shishkin-les-...,Компания «Шишкин Лес» основана в 1998 году. За...,"Сахар, Кофе, Чай, Вода, Питьевая вода",Москва (Города федерального значения),Производитель,"Москва,",NaN,NaN,NaN,+7(495) 258-25-78,info@cone-forest.ru,https://cone-forest.ru/,55.625578,37.606392
9,Эвотор,"""На-полке"" - это онлайн-платформа, помогающая ...",https://foodsuppliers.ru/company/evotor,"""На-полке"" - это онлайн-платформа, помогающая ...","Дрожжи, Макароны, Мука, Приправы, Сахар, Соль,...",Москва (Города федерального значения),Поставщик,"Москва, ул. ленинская слобод

In [68]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.scatter_mapbox(
    merged_df,
    lat="Latitude",
    lon="Longitude",
    hover_name="Название",
    zoom=10,
    height=600,
    size=[5] * len(merged_df)
)
fig.add_trace(
    go.Scattermapbox(
        lat=[55.752110],
        lon=[37.694917],
        mode="markers",
        marker=dict(size=12, color="red"),
        name="ЖК Символ",
        hovertext=["Место открытия кофейни"]
    )
)
fig.update_layout(
    mapbox_style="open-street-map",
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)
fig.show()